In [ ]:
import os
import glob
import pyabf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
def plot_selected_sweeps_200ms(abf_path, sweep_numbers=[9, 19, 29], window_ms=1000):
    """
    Plot and save a 200 ms window from specified sweeps in an ABF file as 3 subplots.

    Parameters:
    - abf_path: Path to the ABF file.
    - sweep_numbers: List of sweep numbers to extract.
    - window_ms: Duration of window in milliseconds.
    """
    abf = pyabf.ABF(abf_path)
    ms_per_point = 1000 / abf.dataRate
    points_in_window = int(window_ms / ms_per_point)

    colors = ['black', 'red', 'blue']
    fig, axs = plt.subplots(3, 1, figsize=(8, 6), sharex=True, sharey=True)
    fig.suptitle(f"200 ms Snippets from Sweeps 9, 19, 29\n{os.path.basename(abf_path)}")

    for i, sweep_num in enumerate(sweep_numbers):
        if sweep_num >= abf.sweepCount:
            axs[i].text(0.5, 0.5, f"Sweep {sweep_num} not available", ha='center', va='center')
            axs[i].set_axis_off()
            continue

        abf.setSweep(sweep_num)
        sweepY = abf.sweepY[:points_in_window]
        sweepX = abf.sweepX[:points_in_window] * 1000  # convert to ms

        axs[i].plot(sweepX, sweepY, color=colors[i])
        axs[i].set_ylabel("pA")
        axs[i].set_title(f"Sweep {sweep_num}")

    axs[-1].set_xlabel("Time (ms)")
    plt.tight_layout(rect=[0, 0, 1, 0.95])

    # Save plot
    base_name = os.path.splitext(os.path.basename(abf_path))[0]
    output_path = os.path.join(os.path.dirname(abf_path), f"{base_name}_sweeps_9_19_29.png")
    plt.savefig(output_path, dpi=300)
    plt.close()
    print(f"Saved: {output_path}")

def process_directory_abf_200ms(directory_path):
    """
    Iterate through all ABF files in a directory and save 200 ms window plots from sweeps 9, 19, and 29.
    """
    abf_files = glob.glob(os.path.join(directory_path, "*.abf"))
    if not abf_files:
        print("No ABF files found.")
        return

    for abf_file in abf_files:
        try:
            plot_selected_sweeps_200ms(abf_file)
        except Exception as e:
            print(f"Error processing {abf_file}: {e}")

In [ ]:
process_directory_abf_200ms("./data/ManualGranuleRawABF")